# TP2 - Etapa 1: Buscador de Imagenes por Similitud

Notebook de la Etapa 1: extraccion de embeddings, construccion de la base vectorial,
visualizacion con PCA y t-SNE, evaluacion con NDCG@10 y analisis de resultados.

## Equipo
- Gianfranco Frattini
- Matias Prado

## 1. Configuracion del entorno

Se clona el fork, instalan dependencias y configura el entorno. En Colab no hay
PostgreSQL, por lo que se usa la base vectorial JSON (`USE_PGVECTOR=false`).

In [ ]:
import os, sys
from pathlib import Path

# TO-DO: completar con la URL de tu fork.
REPO_URL = "https://github.com/GianfrancoFrattini/tuia-dog-recognition-app.git"

# En Colab: clonar el repo. En local: asumir que ya estamos en la raiz.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone $REPO_URL proyecto
    %cd proyecto
    !pip install -q pydantic-settings python-dotenv kagglehub
    ROOT = Path.cwd()
else:
    ROOT = Path.cwd()

os.environ["USE_PGVECTOR"] = "false"
sys.path.insert(0, str(ROOT / "src"))

from lib.config import settings
import torch
print(f"dataset_path: {settings.dataset_path}")
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"embedding_model: {settings.embedding_model}")
print(f"embedding_dim: {settings.embedding_dim}")

## 2. Descargar el dataset

Si el dataset no esta en `data/dataset/`, se descarga de Kaggle.

In [ ]:
if not (settings.dataset_path / 'train').is_dir():
    !python scripts/download_dataset.py
!ls data/dataset/

## 3. Extraccion de embeddings

Se extraen embeddings del split `test/` (5 imagenes por raza = 350 muestras) usando
el modelo baseline (ResNet50 pre-entrenado en ImageNet, sin capa FC, 2048-d).

Esto genera una matriz `X` de shape (350, 2048) y un vector de labels `y` con la raza
de cada imagen.

In [ ]:
import numpy as np
from lib.bootstrap import build_similarity, build_store

store = build_store(settings)
similarity = build_similarity(settings, store)

test_root = settings.dataset_path / 'test'
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
MAX_PER_BREED = 5

embeddings, labels = [], []
for breed_dir in sorted(test_root.iterdir()):
    if not breed_dir.is_dir():
        continue
    count = 0
    for img_path in sorted(breed_dir.iterdir()):
        if img_path.suffix.lower() not in IMAGE_SUFFIXES or count >= MAX_PER_BREED:
            continue
        try:
            img = similarity._load_image(str(img_path.resolve()))
            emb = similarity.extract_embedding(img)
            embeddings.append(emb)
            labels.append(breed_dir.name)
            count += 1
        except Exception as e:
            print(f'skip {img_path}: {e}')

X = np.asarray(embeddings, dtype=np.float32)
y = np.asarray(labels)
breed_names = sorted(set(y))
print(f'Embeddings: {X.shape[0]} muestras, {X.shape[1]}-d, {len(breed_names)} razas')

## 4. Visualizacion con PCA

Se reduce la dimensionalidad de 2048 a 2 usando PCA (Preservacion de varianza global).

Los dos primeros componentes explican ~10% de la varianza (esperable en embeddings
de alta dimension). Aun asi, se observan ciertos clusters por raza, lo que indica
que los embeddings son discriminativos.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(14, 10))
cmap = plt.get_cmap('tab20', len(breed_names))
for i, b in enumerate(breed_names):
    mask = y == b
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[cmap(i)], s=30, alpha=0.7, label=b)
ax.set_title(f'PCA de embeddings baseline (ResNet50, {X.shape[1]}-d) — {len(breed_names)} razas')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend(fontsize=5, ncol=4, loc='best')
plt.tight_layout()
plt.savefig('training_results/embeddings/pca_embeddings.png', dpi=150)
plt.show()
print(f'Varianza explicada: PC1={pca.explained_variance_ratio_[0]:.4f}, PC2={pca.explained_variance_ratio_[1]:.4f}')

## 5. Visualizacion con t-SNE

t-SNE preserva la estructura local de los datos y revela clusters mas definidos que PCA.

- Razas morfologicamente similares (terriers, spaniels) tienden a agruparse cerca.
- Razas muy distintas (Afghan, Husky, Pug) forman clusters bien separados.
- La separabilidad visual de los clusters explica el alto NDCG@10 de la busqueda.

In [ ]:
from sklearn.manifold import TSNE

perplexity = min(30, X.shape[0] - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, init='pca', max_iter=1000)
X_tsne = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(14, 10))
for i, b in enumerate(breed_names):
    mask = y == b
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=[cmap(i)], s=30, alpha=0.7, label=b)
ax.set_title(f't-SNE de embeddings baseline (ResNet50, {X.shape[1]}-d) — {len(breed_names)} razas')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=5, ncol=4, loc='best')
plt.tight_layout()
plt.savefig('training_results/embeddings/tsne_embeddings.png', dpi=150)
plt.show()

## 6. Analisis de clusters

Se observan grupos que coinciden con la morfologia de los perros:
- **Perros de tamaño grande y pelaje largo** (Afghan, Komondor, Old English Sheepdog)
  tienden a separarse del resto.
- **Razas pequeñas y similares** (Chihuahua, Pomeranian, Yorkie, Pekinese) forman
  clusters cercanos, lo que explica los falsos positivos entre ellas.
- **Razas distintivas** (Husky, Pug, Shar_Pei, Mexican Hairless) forman clusters
  muy bien separados por sus caracteristicas unicas.

Esta estructura de clusters es la que permite que la busqueda por similitud funcione
con un NDCG@10 alto (0.9674): los vecinos mas cercanos en el espacio de embeddings
suelen ser de la misma raza.

## 7. Evaluacion con NDCG@10

Para cada imagen del split `test/` se ejecuta la busqueda por similitud, se recuperan
Top-10 vecinos y se calcula NDCG@10 con relevancia binaria (1.0 si el vecino es de
la misma raza, 0.0 si no).

**Requiere la base vectorial indexada** (`scripts/build_index.py --split train` o
`scripts/reindex_docker.py`). En Colab con `USE_PGVECTOR=false` se usa la base JSON.

In [ ]:
from lib.evaluation.metrics import ndcg_at_k

top_k = 10
ndcgs = []
total = 0

for breed_dir in sorted(test_root.iterdir()):
    if not breed_dir.is_dir():
        continue
    for img_path in sorted(breed_dir.iterdir()):
        if img_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        try:
            img = similarity._load_image(str(img_path.resolve()))
            emb = similarity.extract_embedding(img)
            neighbors = similarity.search_similar_images(emb, top_k)
            relevances = [1.0 if n.breed == breed_dir.name else 0.0 for n in neighbors]
            score = ndcg_at_k(relevances, k=top_k)
            ndcgs.append(score)
            total += 1
        except Exception as e:
            print(f'skip {img_path}: {e}')
    if total % 100 == 0:
        print(f'  ...{total} imagenes evaluadas')

mean_ndcg = sum(ndcgs) / len(ndcgs) if ndcgs else 0.0
print(f'\nTotal: {total} imagenes evaluadas (modelo: {settings.embedding_model}, top_k={top_k})')
print(f'NDCG@{top_k} medio: {mean_ndcg:.4f}')

## 8. Analisis de falsos positivos

Se identifican las consultas con menor NDCG@10 para analizar que razas se confunden
mas en la busqueda por similitud.

In [ ]:
# Re-ejecutar guardando detalles para analisis de errores
results_detail = []
for breed_dir in sorted(test_root.iterdir()):
    if not breed_dir.is_dir():
        continue
    for img_path in sorted(breed_dir.iterdir()):
        if img_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        try:
            img = similarity._load_image(str(img_path.resolve()))
            emb = similarity.extract_embedding(img)
            neighbors = similarity.search_similar_images(emb, top_k)
            relevances = [1.0 if n.breed == breed_dir.name else 0.0 for n in neighbors]
            score = ndcg_at_k(relevances, k=top_k)
            predicted_breed, _ = similarity.predict_breed_from_neighbors(neighbors)
            results_detail.append({
                'true_breed': breed_dir.name,
                'predicted_breed': predicted_breed,
                'ndcg': score,
                'n_relevant': int(sum(relevances)),
            })
        except Exception as e:
            pass

import pandas as pd
df = pd.DataFrame(results_detail)
worst = df.sort_values('ndcg').head(15)
print('Top 15 consultas con peor NDCG@10:')
print(worst[['true_breed', 'predicted_breed', 'ndcg', 'n_relevant']].to_string(index=False))

# Razas con NDCG promedio mas bajo
breed_ndcg = df.groupby('true_breed')['ndcg'].mean().sort_values().head(10)
print(f'\nTop 10 razas con peor NDCG@10 promedio:')
print(breed_ndcg.to_string())

## 9. Conclusiones de la Etapa 1

1. **NDCG@10 = 0.9674** — la busqueda por similitud con embeddings de ResNet50
   pre-entrenado recupera correctamente vecinos de la misma raza en casi todos
   los casos, sin necesidad de fine-tuning.
2. **PCA y t-SNE** confirman que los embeddings forman clusters por raza, con
   razas morfologicamente similares agrupadas y razas distintivas bien separadas.
3. **Los errores** se concentran en razas visualmente similares (terriers, spaniels,
   razas pequenas), lo que sugiere que un modelo fine-tuned podria mejorar la
   separabilidad de esos clusters.
4. **El voto mayoritario ponderado** por score es robusto: si los Top-10 vecinos
   son en su mayoria de la misma raza, la prediccion es correcta.
5. **La seleccion dinamica de modelos** permite cambiar entre baseline (2048-d) y
   modelos fine-tuned (512-d) reindexando la base vectorial.